Item Classification - Fast/Slow Moving, Volume, Price, Profit Margin
Classifies each battery item into High/Low tiers per metric, using user-adjustable percentile thresholds.

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver
from src.analysis.item_classification import classify_items
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

Widgets for user-adjustable percentile thresholds

In [0]:
dbutils.widgets.text("volume_percentile", "70")
dbutils.widgets.text("velocity_percentile", "70")
dbutils.widgets.text("price_percentile", "70")
dbutils.widgets.text("margin_percentile", "70")

volume_pct = float(dbutils.widgets.get("volume_percentile"))
velocity_pct = float(dbutils.widgets.get("velocity_percentile"))
price_pct = float(dbutils.widgets.get("price_percentile"))
margin_pct = float(dbutils.widgets.get("margin_percentile"))

print(f"Thresholds - Volume: {volume_pct}th pct, Velocity: {velocity_pct}th pct, Price: {price_pct}th pct, Margin: {margin_pct}th pct")

In [0]:
silver = read_silver(blob_service, "live/battery/battery_clean_live.json")
silver["posting_date"] = pd.to_datetime(silver["posting_date"])

print(f"Silver: {silver.shape}")
result = classify_items(silver, volume_pct, velocity_pct, price_pct, margin_pct)
print(result)

Save

In [0]:
buffer = io.BytesIO()
result.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)
blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/analysis/item_classification.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved item_classification.xlsx")